# Section 1: Notebook Setup & Imports

This notebook runs a **500-configuration** XGBoost hyperparameter refinement sweep on Feature Set V3 for the single global model (`derived_8.2`). It expands the neighborhood around the **1.4 SOTA** (d=9 LR=0.005 MCW=8 Est=2500, R²≈0.6584). Parallel training defaults to **6 workers**. Model caches are keyed by **id + CRC32** of the configuration so resume stays valid after config edits. One worker failure does not abort the whole sweep.


In [ ]:
import os
import sys
import random
import time
import json
import zlib
import threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
import xgboost as xgb
from xgboost import XGBRegressor


def find_project_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for cand in candidates:
        if (cand / "data").exists() and (cand / "d_models").exists():
            return cand
    raise FileNotFoundError("Could not locate repo root containing 'data' and 'd_models'")


PROJECT_ROOT = find_project_root()
print(f"Project root found: {PROJECT_ROOT}")

sys_path_root = str(PROJECT_ROOT)
if sys_path_root not in sys.path:
    sys.path.append(sys_path_root)

out_dir = PROJECT_ROOT / "notebooks/experiment/derived_8.2-hyperparameters-1.5"
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {out_dir}")

import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print(f"Random seed set to {SEED}")

PARALLEL_WORKERS = int(os.environ.get("XGB_PARALLEL_WORKERS", "6"))
print(f"Parallel workers: {PARALLEL_WORKERS} (override with XGB_PARALLEL_WORKERS)")

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

try:
    dummy = xgb.XGBRegressor(n_estimators=1, device="cuda")
    dummy.fit(np.array([[1.0]]), np.array([1.0]))
    XGB_DEVICE = "cuda"
    print("XGBoost CUDA support verified and enabled.")
except Exception as e:
    XGB_DEVICE = "cpu"
    print(f"XGBoost CUDA test failed ({e}). Falling back to CPU.")

print("Setup complete. Using device:", XGB_DEVICE)

# Section 2: Data Loading & Preprocessing

Load the derived_8.2 train / val / test splits, parse dates, and form the **trainval** matrix used for all boosters. Evaluation remains on the held-out test split for protocol continuity with 1.3 / 1.4.

In [ ]:
TRAIN_PATH = PROJECT_ROOT / "data/splits/derived_8.2/train.csv"
VAL_PATH = PROJECT_ROOT / "data/splits/derived_8.2/val.csv"
TEST_PATH = PROJECT_ROOT / "data/splits/derived_8.2/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Dataset splits loaded:")
print(f"  train: {train_df.shape}")
print(f"  val:   {val_df.shape}")
print(f"  test:  {test_df.shape}")

for df in [train_df, val_df, test_df]:
    df["date"] = pd.to_datetime(df["date"])
    df["month"] = df["date"].dt.month.astype(int)
    df["year"] = df["date"].dt.year.astype(float)

trainval_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)
print(f"  trainval (concatenated): {trainval_df.shape}")

# Section 3: Feature Set V3 & 500 Hyperparameter Configurations

Load `OVERALL_SELECTED_FEATURES_V3` and build a **deterministic** 500-config search space around the 1.4 SOTA. Groups: **R** references (6), **A** depth×LR×MCW (216), **B** L2×L1 on new backbone (64), **C** sampling (36), **D** estimator budgets (48), **E** depth-11 / gamma / bin / leaf / champions (130). Lean `n_estimators` scale with learning rate.

In [ ]:
import importlib.util

metadata_path = PROJECT_ROOT / "data/splits/derived_8.2/dataset_metadata.py"
spec = importlib.util.spec_from_file_location("dataset_metadata", metadata_path)
dataset_metadata = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dataset_metadata)

FEATURE_SET_V3 = dataset_metadata.OVERALL_SELECTED_FEATURES_V3
TARGET_COL = "soil_moisture_5cm"
SOTA_R2_1_3_LITE = 0.655063  # 1.3-lite Model 4 peak
SOTA_R2_1_4 = 0.658356       # 1.4 Model 13 peak (target to beat)
SOTA_R2_TARGET = SOTA_R2_1_4

print(f"Feature Set V3 count: {len(FEATURE_SET_V3)}")
print(f"1.3-lite SOTA R2: {SOTA_R2_1_3_LITE}")
print(f"1.4 SOTA R2 (target): {SOTA_R2_1_4}")

# Lean / extended n_estimators schedule
LEAN_EST = {
    0.003: 3000,
    0.004: 2800,
    0.005: 2500,
    0.006: 2300,
    0.007: 2100,
    0.008: 2000,
    0.01: 1500,
    0.012: 1400,
    0.015: 1200,
    0.02: 1000,
    0.04: 1500,
}


def lean_estimators(lr: float) -> int:
    if lr in LEAN_EST:
        return LEAN_EST[lr]
    keys = sorted(LEAN_EST.keys())
    nearest = min(keys, key=lambda k: abs(k - lr))
    return LEAN_EST[nearest]


MODELS_CONFIG = []

# ---------------------------------------------------------------------------
# Group R — References (6)
# ---------------------------------------------------------------------------
MODELS_CONFIG.extend([
    {
        "id": 0, "group": "R", "name": "Baseline (MAE)",
        "objective": "reg:absoluteerror", "max_depth": 8, "min_child_weight": 2,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.04, "n_estimators": 1500,
    },
    {
        "id": 1, "group": "R", "name": "Baseline (MSE)",
        "objective": "reg:squarederror", "max_depth": 8, "min_child_weight": 2,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.04, "n_estimators": 1500,
    },
    {
        "id": 2, "group": "R", "name": "1.3-lite SOTA Control",
        "objective": "reg:squarederror", "max_depth": 8, "min_child_weight": 10,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.01, "n_estimators": 1500,
    },
    {
        "id": 3, "group": "R", "name": "1.4 SOTA Control",
        "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.005, "n_estimators": 2500,
    },
    {
        "id": 4, "group": "R", "name": "1.4 SOTA Est=3000",
        "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.005, "n_estimators": 3000,
    },
    {
        "id": 5, "group": "R", "name": "1.4 Fast Near-SOTA",
        "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 10,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.01, "n_estimators": 1500,
    },
])
model_id = 6

# ---------------------------------------------------------------------------
# Group A — Depth x LR x MCW primary grid (216)
# ---------------------------------------------------------------------------
for depth in [8, 9, 10]:
    for lr in [0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.01, 0.012]:
        for mcw in [3, 4, 5, 6, 8, 10, 12, 15, 20]:
            est = lean_estimators(lr)
            MODELS_CONFIG.append({
                "id": model_id,
                "group": "A",
                "name": f"A MSE d={depth} LR={lr} MCW={mcw} Est={est}",
                "objective": "reg:squarederror",
                "max_depth": depth,
                "min_child_weight": mcw,
                "reg_lambda": 1.5,
                "reg_alpha": 0.03,
                "subsample": 0.9,
                "colsample_bytree": 0.8,
                "learning_rate": lr,
                "n_estimators": est,
            })
            model_id += 1

# ---------------------------------------------------------------------------
# Group B — L2 x L1 on NEW (1.4) SOTA backbone (64)
# ---------------------------------------------------------------------------
for l2 in [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0]:
    for l1 in [0.0, 0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.2]:
        MODELS_CONFIG.append({
            "id": model_id,
            "group": "B",
            "name": f"B MSE L2={l2} L1={l1} (d9 LR0.005 MCW8)",
            "objective": "reg:squarederror",
            "max_depth": 9,
            "min_child_weight": 8,
            "reg_lambda": l2,
            "reg_alpha": l1,
            "subsample": 0.9,
            "colsample_bytree": 0.8,
            "learning_rate": 0.005,
            "n_estimators": 2500,
        })
        model_id += 1

# ---------------------------------------------------------------------------
# Group C — subsample x colsample on new backbone (36)
# ---------------------------------------------------------------------------
for sub in [0.75, 0.8, 0.85, 0.9, 0.95, 1.0]:
    for col in [0.65, 0.7, 0.75, 0.8, 0.85, 0.9]:
        MODELS_CONFIG.append({
            "id": model_id,
            "group": "C",
            "name": f"C MSE sub={sub} col={col} (d9 LR0.005 MCW8)",
            "objective": "reg:squarederror",
            "max_depth": 9,
            "min_child_weight": 8,
            "reg_lambda": 1.5,
            "reg_alpha": 0.03,
            "subsample": sub,
            "colsample_bytree": col,
            "learning_rate": 0.005,
            "n_estimators": 2500,
        })
        model_id += 1

# ---------------------------------------------------------------------------
# Group D — Estimator budget probes (48)
# ---------------------------------------------------------------------------
for depth in [9, 10]:
    for lr in [0.003, 0.005, 0.008, 0.01]:
        for est in [1500, 2000, 2500, 3000, 4000, 5000]:
            MODELS_CONFIG.append({
                "id": model_id,
                "group": "D",
                "name": f"D d={depth} LR={lr} Est={est} MCW=8",
                "objective": "reg:squarederror",
                "max_depth": depth,
                "min_child_weight": 8,
                "reg_lambda": 1.5,
                "reg_alpha": 0.03,
                "subsample": 0.9,
                "colsample_bytree": 0.8,
                "learning_rate": lr,
                "n_estimators": est,
            })
            model_id += 1

# ---------------------------------------------------------------------------
# Group E — Depth-11, gamma, bin, leaf, champions (130)
# ---------------------------------------------------------------------------
# E1: gamma on 1.4 SOTA backbone (8)
for gamma in [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]:
    MODELS_CONFIG.append({
        "id": model_id,
        "group": "E",
        "name": f"E Gamma={gamma} (d9 LR0.005 MCW8)",
        "objective": "reg:squarederror",
        "max_depth": 9,
        "min_child_weight": 8,
        "reg_lambda": 1.5,
        "reg_alpha": 0.03,
        "subsample": 0.9,
        "colsample_bytree": 0.8,
        "learning_rate": 0.005,
        "n_estimators": 2500,
        "gamma": gamma,
    })
    model_id += 1

# E2: max_bin x depth (10)
for max_bin in [64, 128, 256, 512, 1024]:
    for depth in [9, 10]:
        MODELS_CONFIG.append({
            "id": model_id,
            "group": "E",
            "name": f"E Bin={max_bin} d={depth} LR0.005 MCW8",
            "objective": "reg:squarederror",
            "max_bin": max_bin,
            "max_depth": depth,
            "min_child_weight": 8,
            "reg_lambda": 1.5,
            "reg_alpha": 0.03,
            "subsample": 0.9,
            "colsample_bytree": 0.8,
            "learning_rate": 0.005,
            "n_estimators": 2500,
        })
        model_id += 1

# E3: leaf-wise hybrids (48)
for leaves in [63, 127, 255, 511]:
    for depth in [9, 10]:
        for mcw in [6, 8, 10]:
            for max_bin in [128, 256]:
                MODELS_CONFIG.append({
                    "id": model_id,
                    "group": "E",
                    "name": f"E Leaf={leaves} d={depth} MCW={mcw} Bin={max_bin}",
                    "objective": "reg:squarederror",
                    "grow_policy": "lossguide",
                    "max_leaves": leaves,
                    "max_depth": depth,
                    "min_child_weight": mcw,
                    "max_bin": max_bin,
                    "reg_lambda": 1.5,
                    "reg_alpha": 0.03,
                    "subsample": 0.9,
                    "colsample_bytree": 0.8,
                    "learning_rate": 0.005,
                    "n_estimators": 2500,
                })
                model_id += 1

# E4: depth-11 grid (30)
for lr in [0.003, 0.004, 0.005, 0.006, 0.008, 0.01]:
    for mcw in [5, 6, 8, 10, 12]:
        est = lean_estimators(lr)
        MODELS_CONFIG.append({
            "id": model_id,
            "group": "E",
            "name": f"E d=11 LR={lr} MCW={mcw} Est={est}",
            "objective": "reg:squarederror",
            "max_depth": 11,
            "min_child_weight": mcw,
            "reg_lambda": 1.5,
            "reg_alpha": 0.03,
            "subsample": 0.9,
            "colsample_bytree": 0.8,
            "learning_rate": lr,
            "n_estimators": est,
        })
        model_id += 1

# E5: L2 x MCW joint on d=9 LR=0.005 Est=2500 (12)
for l2 in [0.25, 0.5, 1.0, 1.5]:
    for mcw in [6, 8, 10]:
        MODELS_CONFIG.append({
            "id": model_id,
            "group": "E",
            "name": f"E L2={l2} MCW={mcw} (d9 LR0.005)",
            "objective": "reg:squarederror",
            "max_depth": 9,
            "min_child_weight": mcw,
            "reg_lambda": l2,
            "reg_alpha": 0.03,
            "subsample": 0.9,
            "colsample_bytree": 0.8,
            "learning_rate": 0.005,
            "n_estimators": 2500,
        })
        model_id += 1

# E6: champion multi-ingredient combos (22)
champion_specs = [
    {"name": "E Champ d9 LR0.005 MCW8 L2=0.5 L1=0.05",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 0.5, "reg_alpha": 0.05, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d9 LR0.005 MCW8 L2=0.5 L1=0.03",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 0.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d9 LR0.004 MCW8 L2=0.5 L1=0.03 Est2800",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 0.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.004, "n_estimators": 2800},
    {"name": "E Champ d9 LR0.004 MCW8 L2=1.5 L1=0.03 Est2800",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.004, "n_estimators": 2800},
    {"name": "E Champ d10 LR0.005 MCW8 L2=0.5 L1=0.05",
     "objective": "reg:squarederror", "max_depth": 10, "min_child_weight": 8,
     "reg_lambda": 0.5, "reg_alpha": 0.05, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d10 LR0.005 MCW8 L2=0.5 L1=0.03",
     "objective": "reg:squarederror", "max_depth": 10, "min_child_weight": 8,
     "reg_lambda": 0.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d9 LR0.005 MCW6 L2=1.0 L1=0.02",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 6,
     "reg_lambda": 1.0, "reg_alpha": 0.02, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d9 LR0.005 MCW6 L2=0.5 L1=0.05",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 6,
     "reg_lambda": 0.5, "reg_alpha": 0.05, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d10 LR0.004 MCW8 Est2800",
     "objective": "reg:squarederror", "max_depth": 10, "min_child_weight": 8,
     "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.004, "n_estimators": 2800},
    {"name": "E Champ d10 LR0.003 MCW8 Est3000",
     "objective": "reg:squarederror", "max_depth": 10, "min_child_weight": 8,
     "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.003, "n_estimators": 3000},
    {"name": "E Champ d9 LR0.005 MCW8 L2=0.25 L1=0.05",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 0.25, "reg_alpha": 0.05, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d9 LR0.006 MCW8 L2=0.5 L1=0.05 Est2300",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 0.5, "reg_alpha": 0.05, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.006, "n_estimators": 2300},
    {"name": "E Champ d9 LR0.005 MCW8 sub0.95 col0.8",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.95, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d9 LR0.005 MCW8 sub0.9 col0.75",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.75,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d9 LR0.005 MCW8 Bin128",
     "objective": "reg:squarederror", "max_bin": 128, "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d9 LR0.005 MCW8 Bin64",
     "objective": "reg:squarederror", "max_bin": 64, "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d10 LR0.005 MCW6 L2=0.5 L1=0.05",
     "objective": "reg:squarederror", "max_depth": 10, "min_child_weight": 6,
     "reg_lambda": 0.5, "reg_alpha": 0.05, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d11 LR0.005 MCW8 L2=0.5 L1=0.05",
     "objective": "reg:squarederror", "max_depth": 11, "min_child_weight": 8,
     "reg_lambda": 0.5, "reg_alpha": 0.05, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ d9 LR0.005 MCW8 L2=0.5 L1=0.05 Est3000",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 0.5, "reg_alpha": 0.05, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 3000},
    {"name": "E Champ d9 LR0.005 MCW8 L2=0.5 L1=0.05 Est4000",
     "objective": "reg:squarederror", "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 0.5, "reg_alpha": 0.05, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 4000},
    {"name": "E Champ Leaf127 Bin128 d9 MCW8 LR0.005",
     "objective": "reg:squarederror", "grow_policy": "lossguide", "max_leaves": 127, "max_bin": 128,
     "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
    {"name": "E Champ Leaf255 Bin128 d9 MCW8 LR0.005",
     "objective": "reg:squarederror", "grow_policy": "lossguide", "max_leaves": 255, "max_bin": 128,
     "max_depth": 9, "min_child_weight": 8,
     "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
     "learning_rate": 0.005, "n_estimators": 2500},
]
assert len(champion_specs) == 22, f"Expected 22 champions, got {len(champion_specs)}"
for spec in champion_specs:
    MODELS_CONFIG.append({"id": model_id, "group": "E", **spec})
    model_id += 1

# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------
EXPECTED_GROUPS = Counter({"R": 6, "A": 216, "B": 64, "C": 36, "D": 48, "E": 130})
group_counts = Counter(c["group"] for c in MODELS_CONFIG)
ids = [c["id"] for c in MODELS_CONFIG]

assert len(MODELS_CONFIG) == 500, f"Expected 500 configs, got {len(MODELS_CONFIG)}"
assert len(ids) == len(set(ids)), "Duplicate model ids"
assert ids == list(range(500)), "Model ids must be contiguous 0..499"
assert group_counts == EXPECTED_GROUPS, f"Group counts mismatch: {group_counts} vs {EXPECTED_GROUPS}"

print("Group counts:", dict(group_counts))
print(f"Total model configurations: {len(MODELS_CONFIG)}")
print("1.3-lite control (id=2):", {k: MODELS_CONFIG[2][k] for k in (
    "objective", "max_depth", "min_child_weight", "reg_lambda", "reg_alpha",
    "subsample", "colsample_bytree", "learning_rate", "n_estimators",
)})
print("1.4 SOTA control (id=3):", {k: MODELS_CONFIG[3][k] for k in (
    "objective", "max_depth", "min_child_weight", "reg_lambda", "reg_alpha",
    "subsample", "colsample_bytree", "learning_rate", "n_estimators",
)})

# Section 4: Parallel Training & Timing Loop

Train all 500 configs with a **thread pool** (default 6 workers). Each model is cached as `models/xgb_model_{id}_{crc32}.json` where CRC32 is a short hash of the training configuration payload (excluding id/name/group). Resume only hits when both id and CRC match. Worker exceptions are caught per-future so one failure cannot abort the sweep; failures land in `failed_configs.csv` and the cell raises after saving successful models.

In [ ]:
y_trainval = np.asarray(trainval_df[TARGET_COL]).ravel()
y_test = np.asarray(test_df[TARGET_COL]).ravel()
X_trainval = trainval_df[FEATURE_SET_V3]
X_test = test_df[FEATURE_SET_V3]

models_dir = out_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

test_predictions_df = test_df[["date", "year", "month", TARGET_COL]].copy()
trained_models = {}
evals_results = {}
_save_lock = threading.Lock()
_print_lock = threading.Lock()

OPTIONAL_PARAMS = [
    "grow_policy", "max_leaves", "colsample_bylevel", "colsample_bynode",
    "max_bin", "huber_slope", "gamma",
]

FINGERPRINT_EXCLUDE = {"id", "name", "group", "train_time_s", "inference_time_s"}


def config_crc32(config: dict) -> str:
    """Short CRC32 hex of training-relevant config fields (8 hex chars)."""
    payload = {
        k: config[k]
        for k in sorted(config.keys())
        if k not in FINGERPRINT_EXCLUDE
    }
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"), default=str)
    return f"{zlib.crc32(raw.encode('utf-8')) & 0xFFFFFFFF:08x}"


def model_paths(config):
    crc = config_crc32(config)
    mid = config["id"]
    model_path = models_dir / f"xgb_model_{mid}_{crc}.json"
    meta_path = models_dir / f"xgb_model_{mid}_{crc}_meta.json"
    return crc, model_path, meta_path


def build_params(config):
    params = {
        "objective": config["objective"],
        "random_state": SEED,
        "n_jobs": 1,  # avoid oversubscription under parallel model training
        "subsample": config["subsample"],
        "colsample_bytree": config["colsample_bytree"],
        "max_depth": config["max_depth"],
        "min_child_weight": config["min_child_weight"],
        "n_estimators": config["n_estimators"],
        "learning_rate": config["learning_rate"],
        "reg_lambda": config["reg_lambda"],
        "reg_alpha": config["reg_alpha"],
        "device": XGB_DEVICE,
    }
    for param_name in OPTIONAL_PARAMS:
        if param_name in config:
            params[param_name] = config[param_name]
    return params


def load_booster(model_path: Path) -> XGBRegressor:
    """Load booster; tolerate UBJSON content stored under a .json suffix.

    XGBoost 3 defaults to UBJSON when the path does not end in ``.json``/``.ubj``.
    Early 1.5 runs saved via ``*.json.tmp`` (UBJSON) then renamed to ``*.json``,
    so naive load_model fails. Detect content and load via a temp ``.ubj`` path.
    """
    import shutil
    import tempfile

    model = XGBRegressor()
    head = model_path.read_bytes()[:8]
    # Text JSON starts with '{' then a quote/space; UBJSON object starts with '{L'
    is_ubjson = head[:2] == b"{L" or (len(head) >= 2 and head[0:1] == b"{" and head[1:2] != b'"')
    if is_ubjson:
        with tempfile.NamedTemporaryFile(suffix=".ubj", delete=False) as tf:
            tpath = Path(tf.name)
        try:
            shutil.copyfile(model_path, tpath)
            model.load_model(str(tpath))
        finally:
            tpath.unlink(missing_ok=True)
    else:
        model.load_model(str(model_path))
    return model


def save_booster(model: XGBRegressor, model_path: Path) -> None:
    """Atomic save with a temp path that still ends in ``.json`` (forces JSON format)."""
    tmp_model = model_path.parent / f".{model_path.stem}.writing.json"
    model.save_model(str(tmp_model))
    tmp_model.replace(model_path)


def train_one(config):
    """Train or load a single configuration. Returns (config_id, result_dict)."""
    model_id = config["id"]
    model_name = config["name"]
    crc, model_path, meta_path = model_paths(config)
    params = build_params(config)

    if model_path.exists() and meta_path.exists():
        with open(meta_path, "r") as f:
            meta = json.load(f)
        # Belt-and-suspenders: reject stale meta with wrong CRC
        if meta.get("config_crc32", crc) != crc:
            with _print_lock:
                print(f"[{model_id}] CRC mismatch in meta — retraining {model_name}...")
        else:
            try:
                with _print_lock:
                    print(f"[{model_id}] Loading {model_name} (crc={crc}) from disk...")
                model = load_booster(model_path)
                train_time = meta["train_time_s"]
                inference_time = meta["inference_time_s"]
                evals = meta.get("evals_result", {})
                preds = np.asarray(model.predict(X_test)).ravel()
                return model_id, {
                    "model": model,
                    "preds": preds,
                    "train_time_s": train_time,
                    "inference_time_s": inference_time,
                    "evals": evals,
                    "name": model_name,
                    "crc": crc,
                    "loaded": True,
                }
            except Exception as e:
                with _print_lock:
                    print(f"[{model_id}] Load failed ({type(e).__name__}: {e}) — retraining...")

    with _print_lock:
        print(f"[{model_id}] Training {model_name} (crc={crc})...")

    model = XGBRegressor(**params)
    t0 = time.perf_counter()
    model.fit(
        X_trainval, y_trainval,
        eval_set=[(X_trainval, y_trainval), (X_test, y_test)],
        verbose=False,
    )
    t1 = time.perf_counter()
    train_time = t1 - t0

    t2 = time.perf_counter()
    preds = np.asarray(model.predict(X_test)).ravel()
    t3 = time.perf_counter()
    inference_time = t3 - t2

    evals = model.evals_result()
    meta = {
        "train_time_s": train_time,
        "inference_time_s": inference_time,
        "params": {k: v for k, v in params.items() if k != "device"},
        "evals_result": evals,
        "name": model_name,
        "group": config.get("group"),
        "config_crc32": crc,
        "config_id": model_id,
    }

    # Atomic save under lock
    with _save_lock:
        save_booster(model, model_path)
        tmp_meta = meta_path.parent / f".{meta_path.stem}.writing.json"
        with open(tmp_meta, "w") as f:
            json.dump(meta, f, indent=2)
        tmp_meta.replace(meta_path)

    with _print_lock:
        print(f"[{model_id}] Done in {train_time:.2f}s (infer {inference_time:.4f}s, crc={crc})")

    return model_id, {
        "model": model,
        "preds": preds,
        "train_time_s": train_time,
        "inference_time_s": inference_time,
        "evals": evals,
        "name": model_name,
        "crc": crc,
        "loaded": False,
    }


# Preview a couple of cache keys
for _preview_id in (0, 3):
    _cfg = MODELS_CONFIG[_preview_id]
    _crc, _mp, _ = model_paths(_cfg)
    print(f"Cache key preview id={_preview_id}: {_mp.name}")

print(f"\nStarting parallel training for {len(MODELS_CONFIG)} configurations "
      f"with {PARALLEL_WORKERS} workers...\n")
wall_t0 = time.perf_counter()
results_by_id = {}
failed = []
completed = 0

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
    futures = {executor.submit(train_one, cfg): cfg for cfg in MODELS_CONFIG}
    for fut in as_completed(futures):
        cfg = futures[fut]
        mid = cfg["id"]
        try:
            mid, result = fut.result()
            results_by_id[mid] = result
        except Exception as e:
            failed.append({
                "id": mid,
                "name": cfg.get("name"),
                "group": cfg.get("group"),
                "error": f"{type(e).__name__}: {e}",
            })
            with _print_lock:
                print(f"[{mid}] FAILED: {type(e).__name__}: {e}")
        completed += 1
        if completed % 25 == 0 or completed == len(MODELS_CONFIG):
            elapsed = time.perf_counter() - wall_t0
            rate = completed / elapsed if elapsed > 0 else 0
            remaining = (len(MODELS_CONFIG) - completed) / rate if rate > 0 else float("nan")
            n_loaded = sum(1 for r in results_by_id.values() if r.get("loaded"))
            with _print_lock:
                print(f"  Progress: {completed}/{len(MODELS_CONFIG)} "
                      f"({elapsed:.1f}s elapsed, ~{remaining:.1f}s remaining, "
                      f"{len(failed)} failed, {n_loaded} loaded from cache)")

wall_t1 = time.perf_counter()
print(f"\nProcessed {completed} configs in {wall_t1 - wall_t0:.1f}s wall time "
      f"({len(results_by_id)} ok, {len(failed)} failed).")

if failed:
    failed_df = pd.DataFrame(failed)
    failed_path = out_dir / "failed_configs.csv"
    failed_df.to_csv(failed_path, index=False)
    print(f"WARNING: {len(failed)} configs failed. See {failed_path}")
else:
    failed_path = out_dir / "failed_configs.csv"
    if failed_path.exists():
        failed_path.unlink()

# Assemble predictions and per-config timing in deterministic id order
for config in MODELS_CONFIG:
    mid = config["id"]
    if mid not in results_by_id:
        test_predictions_df[f"pred_{mid}"] = np.nan
        config["train_time_s"] = float("nan")
        config["inference_time_s"] = float("nan")
        trained_models[mid] = None
        evals_results[mid] = {}
        continue
    result = results_by_id[mid]
    test_predictions_df[f"pred_{mid}"] = result["preds"]
    config["train_time_s"] = result["train_time_s"]
    config["inference_time_s"] = result["inference_time_s"]
    trained_models[mid] = result["model"]
    evals_results[mid] = result["evals"]

test_predictions_df.to_csv(out_dir / "test_predictions.csv", index=False)
print(f"Saved test predictions to: {out_dir / 'test_predictions.csv'}")

# Step-by-step loss curves (test split monitored as validation_1)
max_steps = 0
for mid in evals_results:
    if not evals_results[mid]:
        continue
    metric_key = list(evals_results[mid]["validation_0"].keys())[0]
    max_steps = max(max_steps, len(evals_results[mid]["validation_0"][metric_key]))

loss_data = {"step": list(range(max_steps))}
for mid in sorted(evals_results.keys()):
    evals = evals_results[mid]
    if not evals:
        loss_data[f"train_loss_model_{mid}"] = [np.nan] * max_steps
        loss_data[f"test_loss_model_{mid}"] = [np.nan] * max_steps
        continue
    metric_key = list(evals["validation_0"].keys())[0]
    train_loss = evals["validation_0"][metric_key]
    test_loss = evals["validation_1"][metric_key]
    loss_data[f"train_loss_model_{mid}"] = list(train_loss) + [np.nan] * (max_steps - len(train_loss))
    loss_data[f"test_loss_model_{mid}"] = list(test_loss) + [np.nan] * (max_steps - len(test_loss))

loss_df = pd.DataFrame(loss_data)
loss_df.to_csv(out_dir / "loss_curves.csv", index=False)
print(f"Saved step-by-step loss history for {len(MODELS_CONFIG)} models to: {out_dir / 'loss_curves.csv'}")

if failed:
    raise RuntimeError(
        f"{len(failed)} configs failed (see failed_configs.csv). "
        f"{len(results_by_id)} succeeded and were saved — re-run to resume."
    )


# Section 5: Overall Metrics Summary

Compute R², RMSE, ubRMSE, Bias, MAE, median absolute error, and Pearson correlation for every successful config on the held-out test set. Write `metrics_summary.csv` and print the top-20 leaderboard by R².

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    if np.any(np.isnan(y_pred)):
        return {
            "R2": float("nan"), "RMSE": float("nan"), "ubRMSE": float("nan"),
            "Bias": float("nan"), "MAE": float("nan"), "Med|Err|": float("nan"),
            "Pearson": float("nan"),
        }
    err = y_true - y_pred
    ae = np.abs(err)
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    ubrmse = np.sqrt(np.mean(((y_true - np.mean(y_true)) - (y_pred - np.mean(y_pred))) ** 2))
    bias = np.mean(err)
    med_ae = np.median(ae)
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        pearson = float("nan")
    else:
        pearson = np.corrcoef(y_true, y_pred)[0, 1]
    return {
        "R2": r2, "RMSE": rmse, "ubRMSE": ubrmse, "Bias": bias,
        "MAE": mae, "Med|Err|": med_ae, "Pearson": pearson,
    }


summary_records = []
for config in MODELS_CONFIG:
    model_id = config["id"]
    preds = test_predictions_df[f"pred_{model_id}"]
    metrics = compute_metrics(y_test, preds)
    summary_records.append({
        "id": model_id,
        "group": config.get("group", ""),
        "Configuration": config["name"],
        **metrics,
        "Train Time (s)": config["train_time_s"],
        "Inference Time (s)": config["inference_time_s"],
    })

summary_df = pd.DataFrame(summary_records)
cols_order = [
    "id", "group", "Configuration", "R2", "RMSE", "ubRMSE", "Bias", "MAE",
    "Med|Err|", "Pearson", "Train Time (s)", "Inference Time (s)",
]
summary_df = summary_df[cols_order]
summary_df.to_csv(out_dir / "metrics_summary.csv", index=False)

print("===== TOP 20 BY R2 =====")
top20 = summary_df.sort_values("R2", ascending=False).head(20)
print(top20.to_string(index=False, formatters={
    "R2": "{:,.4f}".format,
    "RMSE": "{:,.4f}".format,
    "ubRMSE": "{:,.4f}".format,
    "Bias": "{:+,.4f}".format,
    "MAE": "{:,.4f}".format,
    "Med|Err|": "{:,.4f}".format,
    "Pearson": "{:,.4f}".format,
    "Train Time (s)": "{:,.2f}".format,
    "Inference Time (s)": "{:,.4f}".format,
}))
print(f"\nSaved metrics_summary.csv ({len(summary_df)} rows)")

# Section 6: Year-by-Year Metrics

Break test metrics into calendar years (2023 / 2024 / 2025) so year-specific regressions (e.g. 2024 dips seen in 1.4) remain visible.

In [ ]:
years = sorted(test_df["year"].dropna().unique())
year_records = []
for config in MODELS_CONFIG:
    mid = config["id"]
    preds = test_predictions_df[f"pred_{mid}"].values
    for year in years:
        mask = test_df["year"].values == year
        if mask.sum() == 0:
            continue
        m = compute_metrics(y_test[mask], preds[mask])
        year_records.append({
            "id": mid,
            "group": config.get("group", ""),
            "Configuration": config["name"],
            "year": int(year),
            **m,
        })

metrics_by_year_df = pd.DataFrame(year_records)
metrics_by_year_df.to_csv(out_dir / "metrics_by_year.csv", index=False)
print(f"Saved metrics_by_year.csv ({len(metrics_by_year_df)} rows)")
print("Years:", [int(y) for y in years])
print("\n1.4 SOTA control (id=3) by year:")
print(metrics_by_year_df[metrics_by_year_df["id"] == 3][
    ["year", "R2", "RMSE", "MAE"]
].to_string(index=False))
print("\n1.3-lite control (id=2) by year:")
print(metrics_by_year_df[metrics_by_year_df["id"] == 2][
    ["year", "R2", "RMSE", "MAE"]
].to_string(index=False))

# Section 7: Loss Curves (Selected Models)

Plot train vs test loss for baselines, both SOTA controls, per-group bests, and overall top models. Cap raised to **12** panels so group winners are less likely to be dropped.

In [ ]:
# Select models: baselines, both controls, group bests, overall top
selected_loss_ids = [0, 1, 2, 3]
best_overall = summary_df.sort_values("R2", ascending=False).head(4)["id"].tolist()
for g in ["A", "B", "C", "D", "E"]:
    gdf = summary_df[summary_df["group"] == g]
    if len(gdf):
        selected_loss_ids.append(int(gdf.sort_values("R2", ascending=False).iloc[0]["id"]))
for mid in best_overall:
    if mid not in selected_loss_ids:
        selected_loss_ids.append(int(mid))
# Prefer unique and keep up to 12
seen = set()
uniq = []
for mid in selected_loss_ids:
    if mid not in seen:
        seen.add(mid)
        uniq.append(mid)
selected_loss_ids = uniq[:12]

n_panels = len(selected_loss_ids)
n_cols = 3
n_rows = int(np.ceil(n_panels / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes = np.atleast_1d(axes).flatten()
for i, mid in enumerate(selected_loss_ids):
    config = next(c for c in MODELS_CONFIG if c["id"] == mid)
    evals = evals_results[mid]
    ax = axes[i]
    if not evals:
        ax.set_title(f"[{mid}] {config['name']} (no evals)")
        continue
    metric_key = list(evals["validation_0"].keys())[0]
    ax.plot(evals["validation_0"][metric_key], label=f"Train ({metric_key})", color="#2a9d8f", linewidth=2)
    ax.plot(evals["validation_1"][metric_key], label=f"Test ({metric_key})", color="#e76f51", linewidth=2)
    r2_val = float(summary_df.loc[summary_df["id"] == mid, "R2"].iloc[0])
    ax.set_xlabel("Boosting Round")
    ax.set_ylabel(metric_key.upper())
    ax.set_title(f"[{mid}] {config['name']}\nR2={r2_val:.4f}")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend()
for j in range(n_panels, len(axes)):
    axes[j].axis("off")

plt.suptitle("Train vs Test Loss Curves (Selected Configs)", fontsize=16, fontweight="bold", y=0.995)
plt.tight_layout()
plt.savefig(out_dir / "loss_curves.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved loss_curves.png for models: {selected_loss_ids}")

# Section 8: R² by Year Visualization

Compare overall and per-year R² for top configs against the 1.4 SOTA target line.

In [ ]:
# Top 12 overall + both controls + baselines (unique)
plot_ids = [0, 1, 2, 3]
for mid in summary_df.sort_values("R2", ascending=False)["id"].head(12):
    if int(mid) not in plot_ids:
        plot_ids.append(int(mid))
plot_ids = plot_ids[:16]

rows = []
for mid in plot_ids:
    cfg = next(c for c in MODELS_CONFIG if c["id"] == mid)
    overall_r2 = float(summary_df.loc[summary_df["id"] == mid, "R2"].iloc[0])
    row = {"id": mid, "name": f"[{mid}] {cfg['name']}", "Overall": overall_r2}
    ydf = metrics_by_year_df[metrics_by_year_df["id"] == mid]
    for _, r in ydf.iterrows():
        row[str(int(r["year"]))] = r["R2"]
    rows.append(row)

plot_df = pd.DataFrame(rows)
year_cols = [c for c in plot_df.columns if c not in ("id", "name")]
year_order = ["Overall"] + sorted([c for c in year_cols if c != "Overall"])
plot_df = plot_df.set_index("name")[year_order]

fig, ax = plt.subplots(figsize=(14, max(6, 0.45 * len(plot_df))))
x = np.arange(len(plot_df.columns))
width = 0.8 / max(len(plot_df), 1)
for i, (name, row) in enumerate(plot_df.iterrows()):
    ax.bar(x + i * width, row.values, width=width, label=name[:60])
ax.axhline(SOTA_R2_1_4, color="red", linestyle="--", linewidth=1.5, label=f"1.4 SOTA {SOTA_R2_1_4:.4f}")
ax.axhline(SOTA_R2_1_3_LITE, color="gray", linestyle=":", linewidth=1.2, label=f"1.3-lite {SOTA_R2_1_3_LITE:.4f}")
ax.set_xticks(x + width * (len(plot_df) - 1) / 2)
ax.set_xticklabels(plot_df.columns)
ax.set_ylabel("R²")
ax.set_title("R² Overall and by Year — Top Configs vs SOTA Controls")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(out_dir / "r2_by_year.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved r2_by_year.png for {len(plot_ids)} models")

# Section 9: Residual Plots (Top Configs)

Overall residual scatter for the top 18 configs (ensuring the 1.4 control id=3 is present) and year-stratified residuals for the top 6.

In [ ]:
# --- Overall residuals: top 18 ---
top18_ids = summary_df.sort_values("R2", ascending=False).head(18)["id"].astype(int).tolist()
if 3 not in top18_ids:
    top18_ids[-1] = 3  # ensure 1.4 SOTA control present

fig, axes = plt.subplots(6, 3, figsize=(18, 30))
axes = axes.flatten()
for i, mid in enumerate(top18_ids):
    config = next(c for c in MODELS_CONFIG if c["id"] == mid)
    preds = test_predictions_df[f"pred_{mid}"].values
    res = y_test - preds
    r2_val = float(summary_df.loc[summary_df["id"] == mid, "R2"].iloc[0])
    ax = axes[i]
    ax.scatter(preds, res, s=4, alpha=0.25, c="#3a86c8")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Residual (true - pred)")
    ax.set_title(f"[{mid}] {config['name'][:50]}\nR2={r2_val:.4f}", fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.4)
for j in range(len(top18_ids), len(axes)):
    axes[j].axis("off")
plt.suptitle("Residuals — Top 18 Configurations", fontsize=16, fontweight="bold", y=0.995)
plt.tight_layout()
plt.savefig(out_dir / "residuals_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved residuals_comparison.png")

# --- Residuals by year: top 6 ---
top6_ids = summary_df.sort_values("R2", ascending=False).head(6)["id"].astype(int).tolist()
if 3 not in top6_ids:
    top6_ids = top6_ids[:5] + [3]

unique_years = sorted(int(y) for y in test_df["year"].dropna().unique())
fig, axes = plt.subplots(len(top6_ids), len(unique_years), figsize=(5 * len(unique_years), 3.5 * len(top6_ids)))
if len(top6_ids) == 1:
    axes = np.array([axes])
for i, mid in enumerate(top6_ids):
    config = next(c for c in MODELS_CONFIG if c["id"] == mid)
    preds = test_predictions_df[f"pred_{mid}"].values
    for j, year in enumerate(unique_years):
        ax = axes[i, j]
        mask = test_df["year"].values == year
        res = y_test[mask] - preds[mask]
        r2_y = r2_score(y_test[mask], preds[mask])
        ax.scatter(preds[mask], res, s=4, alpha=0.3, c="#e76f51")
        ax.axhline(0, color="black", linewidth=1)
        ax.set_title(f"[{mid}] {year} R2={r2_y:.4f}", fontsize=9)
        if i == len(top6_ids) - 1:
            ax.set_xlabel("Predicted")
        if j == 0:
            ax.set_ylabel(f"{config['name'][:28]}\nResidual")
        ax.grid(True, linestyle="--", alpha=0.4)
plt.suptitle("Residuals by Year — Top Configs", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(out_dir / "residuals_by_year.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved residuals_by_year.png")

# Section 10: Leaderboard & New SOTA Check

Compare the best model in this sweep against the **1.4 SOTA** (primary target) and the **1.3-lite** control. Persist `leaderboard_top20.csv` and print recommended params for the champion.

In [ ]:
control_1_3 = summary_df.loc[summary_df["id"] == 2].iloc[0]
control_1_4 = summary_df.loc[summary_df["id"] == 3].iloc[0]
sota_control_r2_1_3 = float(control_1_3["R2"])
sota_control_r2_1_4 = float(control_1_4["R2"])
best_row = summary_df.sort_values("R2", ascending=False).iloc[0]
best_r2 = float(best_row["R2"])
best_id = int(best_row["id"])

print("=" * 72)
print("LEADERBOARD — derived_8.2-hyperparameters-1.5")
print("=" * 72)
print(f"1.3-lite SOTA target:                {SOTA_R2_1_3_LITE:.6f}")
print(f"1.3-lite control Model 2 (this run): {sota_control_r2_1_3:.6f}")
print(f"1.4 SOTA target:                     {SOTA_R2_1_4:.6f}")
print(f"1.4 control Model 3 (this run):      {sota_control_r2_1_4:.6f}")
print(f"Best model this sweep:               id={best_id}  R2={best_r2:.6f}")
print(f"  Configuration: {best_row['Configuration']}")
print(f"  Group: {best_row['group']}  Train time: {best_row['Train Time (s)']:.2f}s")

if best_r2 > SOTA_R2_1_4 + 1e-6:
    delta = best_r2 - SOTA_R2_1_4
    print(f"\n*** NEW SOTA ***  ΔR2 = +{delta:.6f} over 1.4 target")
elif best_r2 > sota_control_r2_1_4 + 1e-6:
    print(f"\nBest exceeds this-run 1.4 control by +{best_r2 - sota_control_r2_1_4:.6f} "
          f"(but not the published 1.4 target)")
else:
    print("\nNo new SOTA this sweep; 1.4 control remains best or tied.")

print("\n----- Top 20 -----")
print(summary_df.sort_values("R2", ascending=False).head(20)[
    ["id", "group", "Configuration", "R2", "RMSE", "MAE", "Train Time (s)"]
].to_string(index=False, formatters={
    "R2": "{:.6f}".format, "RMSE": "{:.4f}".format, "MAE": "{:.4f}".format,
    "Train Time (s)": "{:.2f}".format,
}))

print("\n----- Best per group -----")
for g in ["R", "A", "B", "C", "D", "E"]:
    gdf = summary_df[summary_df["group"] == g]
    if gdf.empty:
        continue
    br = gdf.sort_values("R2", ascending=False).iloc[0]
    print(f"  {g}: id={int(br['id']):3d}  R2={br['R2']:.6f}  {br['Configuration']}")

best_cfg = next(c for c in MODELS_CONFIG if c["id"] == best_id)
recommend_keys = [
    "objective", "max_depth", "min_child_weight", "reg_lambda", "reg_alpha",
    "subsample", "colsample_bytree", "n_estimators", "learning_rate",
    "grow_policy", "max_leaves", "max_bin", "gamma", "huber_slope",
]
recommend = {k: best_cfg[k] for k in recommend_keys if k in best_cfg}
print("\n----- Recommended params (best model) -----")
print(json.dumps(recommend, indent=2))

leaderboard_path = out_dir / "leaderboard_top20.csv"
summary_df.sort_values("R2", ascending=False).head(20).to_csv(leaderboard_path, index=False)
print(f"\nSaved {leaderboard_path}")